# Precision, Recall & F1 Score

## Overview
Precision, Recall and F1 are the core metrics for evaluating classifiers,
especially when classes are **imbalanced** or when the cost of false positives
and false negatives differs.

### Formulas:
$$Precision = \\frac{TP}{TP + FP} \\quad \\text{(of all predicted positives, how many were correct?)}$$

$$Recall = \\frac{TP}{TP + FN} \\quad \\text{(of all actual positives, how many did we catch?)}$$

$$F1 = \\frac{2 \\cdot Precision \\cdot Recall}{Precision + Recall} \\quad \\text{(harmonic mean)}$$

$$F_{\\beta} = (1 + \\beta^2) \\cdot \\frac{Precision \\cdot Recall}{\\beta^2 \\cdot Precision + Recall}$$

### When to prioritize which metric:
| Situation | Prioritize | Example |
|-----------|-----------|--------|
| FP is costly | Precision | Spam filter (don't block legit emails) |
| FN is costly | Recall | Cancer screening (don't miss patients) |
| Both matter | F1 | General classification |
| FN worse than FP | F2 (beta=2) | Disease detection |
| FP worse than FN | F0.5 (beta=0.5) | Fraud alerts |

---
### Topics Covered
1. Precision & Recall intuition
2. Precision-Recall tradeoff
3. F1, F-beta scores
4. Precision-Recall curve & Average Precision
5. Threshold tuning for Precision vs Recall
6. Multiclass Precision-Recall (macro, micro, weighted)
7. Imbalanced classes — why F1 matters
8. Real-world offline-safe dataset
9. Model comparison
10. Metric selection guide


## 1. Import Libraries

In [ ]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score
)

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#E67E22',
    'success':   '#27AE60',
    'danger':    '#E74C3C',
    'neutral':   '#7F8C8D',
    'tertiary':  '#9B59B6',
}
MODEL_COLORS = ['#2E86AB','#27AE60','#E67E22','#9B59B6','#E74C3C']

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

print('Libraries imported successfully!')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

## 2. Precision & Recall Intuition — Visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Precision diagram ---
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Precision = TP / (TP + FP)\n'
             'Of everything predicted positive, how many ARE positive?',
             fontsize=11, fontweight='bold')

# Predicted positive circle
circle_pred = plt.Circle((5, 5), 3.5, color=COLORS['primary'],
                           alpha=0.2, linewidth=2, fill=True)
circle_pred_edge = plt.Circle((5, 5), 3.5, color=COLORS['primary'],
                                fill=False, linewidth=2.5)
ax.add_patch(circle_pred)
ax.add_patch(circle_pred_edge)
ax.text(5, 9.2, 'All Predicted Positive', ha='center',
        fontsize=10, color=COLORS['primary'], fontweight='bold')

# TP dots (inside, correct)
np.random.seed(1)
for _ in range(18):
    angle = np.random.uniform(0, 2*np.pi)
    r     = np.random.uniform(0, 2.5)
    ax.scatter(5 + r*np.cos(angle), 5 + r*np.sin(angle),
               color=COLORS['success'], s=80, zorder=5)
# FP dots (inside, wrong)
for _ in range(7):
    angle = np.random.uniform(0, 2*np.pi)
    r     = np.random.uniform(0.5, 3.2)
    ax.scatter(5 + r*np.cos(angle), 5 + r*np.sin(angle),
               color=COLORS['danger'], s=80, marker='X', zorder=5)

ax.legend(handles=[
    mpatches.Patch(color=COLORS['success'], label='TP (correctly predicted positive)'),
    mpatches.Patch(color=COLORS['danger'],  label='FP (wrongly predicted positive)'),
], loc='lower center', fontsize=9)
ax.text(5, 0.5, 'Precision = 18/(18+7) = 0.72', ha='center',
        fontsize=12, fontweight='bold', color=COLORS['primary'])

# --- Recall diagram ---
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('Recall = TP / (TP + FN)\n'
              'Of all actual positives, how many did we CATCH?',
              fontsize=11, fontweight='bold')

# All actual positives
circle_act = plt.Circle((5, 5), 3.5, color=COLORS['success'],
                          alpha=0.2, fill=True)
circle_act_edge = plt.Circle((5, 5), 3.5, color=COLORS['success'],
                               fill=False, linewidth=2.5)
ax2.add_patch(circle_act)
ax2.add_patch(circle_act_edge)
ax2.text(5, 9.2, 'All Actual Positives', ha='center',
         fontsize=10, color=COLORS['success'], fontweight='bold')

np.random.seed(2)
for _ in range(18):
    angle = np.random.uniform(0, 2*np.pi)
    r     = np.random.uniform(0, 2.5)
    ax2.scatter(5 + r*np.cos(angle), 5 + r*np.sin(angle),
                color=COLORS['primary'], s=80, zorder=5)
for _ in range(8):
    angle = np.random.uniform(0, 2*np.pi)
    r     = np.random.uniform(0.5, 3.2)
    ax2.scatter(5 + r*np.cos(angle), 5 + r*np.sin(angle),
                color=COLORS['danger'], s=80, marker='X', zorder=5)

ax2.legend(handles=[
    mpatches.Patch(color=COLORS['primary'], label='TP (caught)'),
    mpatches.Patch(color=COLORS['danger'],  label='FN (missed)'),
], loc='lower center', fontsize=9)
ax2.text(5, 0.5, 'Recall = 18/(18+8) = 0.69', ha='center',
         fontsize=12, fontweight='bold', color=COLORS['success'])

plt.suptitle('Precision vs Recall — Visual Intuition',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Precision-Recall Tradeoff

In [ ]:
np.random.seed(42)
X, y = make_classification(
    n_samples=600, n_features=10, n_informative=6,
    n_redundant=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_sc, y_train)
y_prob = model.predict_proba(X_test_sc)[:, 1]

thresholds  = np.linspace(0.01, 0.99, 150)
precisions  = []
recalls     = []
f1s         = []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_t, zero_division=0))
    recalls.append(   recall_score(   y_test, y_t, zero_division=0))
    f1s.append(       f1_score(       y_test, y_t, zero_division=0))

best_f1_idx = np.argmax(f1s)
best_thresh = thresholds[best_f1_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(thresholds, precisions, color=COLORS['primary'],
             linewidth=2.5, label='Precision')
axes[0].plot(thresholds, recalls,    color=COLORS['danger'],
             linewidth=2.5, label='Recall')
axes[0].plot(thresholds, f1s,        color=COLORS['success'],
             linewidth=2.5, label='F1 Score')
axes[0].axvline(best_thresh, color='black', linestyle=':',
                linewidth=2, label=f'Best F1 thresh={best_thresh:.2f}')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision, Recall & F1 vs Threshold')
axes[0].legend()

# Precision vs Recall plot
axes[1].plot(recalls, precisions, color=COLORS['tertiary'],
             linewidth=2.5)
axes[1].scatter(recalls[best_f1_idx], precisions[best_f1_idx],
                color=COLORS['danger'], s=200, zorder=5,
                label=f'Best F1 point\nP={precisions[best_f1_idx]:.3f}'
                      f' R={recalls[best_f1_idx]:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision vs Recall Tradeoff')
axes[1].legend(fontsize=9)

plt.suptitle('Precision-Recall Tradeoff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'At threshold 0.5  : P={precision_score(y_test, model.predict(X_test_sc)):.4f}'
      f'  R={recall_score(y_test, model.predict(X_test_sc)):.4f}'
      f'  F1={f1_score(y_test, model.predict(X_test_sc)):.4f}')
print(f'At best F1 thresh : P={precisions[best_f1_idx]:.4f}'
      f'  R={recalls[best_f1_idx]:.4f}'
      f'  F1={f1s[best_f1_idx]:.4f}')

## 4. F1, F-beta Scores

In [ ]:
# Demonstrate F-beta with different beta values
# beta > 1 -> weights Recall more
# beta < 1 -> weights Precision more

y_pred_default = model.predict(X_test_sc)

beta_values = [0.25, 0.5, 1.0, 2.0, 4.0]
fbeta_scores = [fbeta_score(y_test, y_pred_default, beta=b) for b in beta_values]

p_val = precision_score(y_test, y_pred_default)
r_val = recall_score(   y_test, y_pred_default)

print(f'Precision : {p_val:.4f}')
print(f'Recall    : {r_val:.4f}')
print()
print(f'{"Beta":<8} {"F-beta":<10} {"Interpretation"}')
print('-' * 55)
interps = [
    'Precision 4x more important',
    'Precision 2x more important',
    'Equal (standard F1)',
    'Recall 2x more important',
    'Recall 4x more important',
]
for b, fb, interp in zip(beta_values, fbeta_scores, interps):
    print(f'{b:<8} {fb:<10.4f} {interp}')

plt.figure(figsize=(10, 5))
plt.plot(beta_values, fbeta_scores, 'o-',
         color=COLORS['primary'], linewidth=2.5, markersize=10)
plt.axhline(p_val, color=COLORS['primary'], linestyle='--',
            linewidth=1.5, alpha=0.6, label=f'Precision={p_val:.3f}')
plt.axhline(r_val, color=COLORS['danger'],  linestyle='--',
            linewidth=1.5, alpha=0.6, label=f'Recall={r_val:.3f}')
plt.axvline(1.0, color='black', linestyle=':', linewidth=1.5,
            label='beta=1 (standard F1)')
for b, fb in zip(beta_values, fbeta_scores):
    plt.text(b, fb + 0.008, f'{fb:.3f}', ha='center', fontsize=9)
plt.xlabel('Beta')
plt.ylabel('F-beta Score')
plt.title('F-beta Score vs Beta\n'
          'beta < 1 -> favors Precision  |  beta > 1 -> favors Recall')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Precision-Recall Curve & Average Precision

In [ ]:
prec_curve, rec_curve, pr_thresh = precision_recall_curve(y_test, y_prob)
avg_prec  = average_precision_score(y_test, y_prob)
baseline  = y_test.mean()

# Best F1 on PR curve
f1_curve   = 2*prec_curve*rec_curve / (prec_curve + rec_curve + 1e-9)
best_pr_idx = np.argmax(f1_curve)

# F-beta iso-curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PR Curve with iso-F1 lines
axes[0].plot(rec_curve, prec_curve, color=COLORS['secondary'],
             linewidth=2.5, label=f'PR Curve (AP={avg_prec:.4f})')
axes[0].axhline(baseline, color='black', linestyle='--',
                linewidth=1.2, label=f'Random baseline={baseline:.3f}')
axes[0].scatter(rec_curve[best_pr_idx], prec_curve[best_pr_idx],
                color=COLORS['danger'], s=200, zorder=5,
                label=f'Best F1 point')
axes[0].fill_between(rec_curve, prec_curve, alpha=0.12, color=COLORS['secondary'])

# Iso-F1 curves
for f1_iso in [0.3, 0.5, 0.7, 0.9]:
    r_iso = np.linspace(0.01, 1.0, 100)
    p_iso = f1_iso * r_iso / (2*r_iso - f1_iso + 1e-9)
    valid = (p_iso >= 0) & (p_iso <= 1)
    axes[0].plot(r_iso[valid], p_iso[valid], color='gray',
                 linewidth=0.8, linestyle=':')
    axes[0].text(r_iso[valid][-1] + 0.01, p_iso[valid][-1],
                 f'F1={f1_iso}', fontsize=7, color='gray')

axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve with Iso-F1 Lines')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-0.02, 1.05)
axes[0].set_ylim(-0.02, 1.05)

# AP interpretation
ap_levels = [0.2, 0.4, 0.6, 0.8, 1.0]
ap_labels = ['Poor','Fair','Good','Very Good','Perfect']
ap_colors = ['#E74C3C','#E67E22','#F39C12','#2ECC71','#27AE60']
for i, (lo, hi, lbl, clr) in enumerate(zip(
    [0]+ap_levels[:-1], ap_levels, ap_labels, ap_colors
)):
    axes[1].barh(lbl, hi-lo, left=lo, color=clr,
                 edgecolor='white', height=0.5)
axes[1].axvline(avg_prec, color='black', linewidth=2.5,
                linestyle='--', label=f'Your model AP={avg_prec:.3f}')
axes[1].axvline(baseline, color=COLORS['neutral'], linewidth=1.5,
                linestyle=':', label=f'Random baseline={baseline:.3f}')
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel('Average Precision (AP)')
axes[1].set_title('AP Score Interpretation Guide')
axes[1].legend(fontsize=9)

plt.suptitle('Precision-Recall Curve & Average Precision',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Average Precision (AP) : {avg_prec:.4f}')
print(f'Baseline (class ratio) : {baseline:.4f}')
print(f'Best F1 on PR curve    : {f1_curve[best_pr_idx]:.4f}')
if best_pr_idx < len(pr_thresh):
    print(f'At threshold           : {pr_thresh[best_pr_idx]:.4f}')

## 6. Threshold Tuning for Specific Goals

In [ ]:
# Use case 1: Medical screening (maximize Recall >= 0.95)
# Use case 2: Fraud alert (maximize Precision >= 0.90)
# Use case 3: Maximize F1

goals = {
    'Max F1':              None,
    'Recall >= 0.90':      None,
    'Precision >= 0.90':   None,
    'Balanced (P=R)':      None,
}

thresh_grid = np.linspace(0.01, 0.99, 500)
P_grid = [precision_score(y_test, (y_prob>=t).astype(int), zero_division=0)
          for t in thresh_grid]
R_grid = [recall_score(   y_test, (y_prob>=t).astype(int), zero_division=0)
          for t in thresh_grid]
F_grid = [f1_score(       y_test, (y_prob>=t).astype(int), zero_division=0)
          for t in thresh_grid]
P_arr = np.array(P_grid)
R_arr = np.array(R_grid)
F_arr = np.array(F_grid)

# Best F1
idx_f1    = np.argmax(F_arr)
goals['Max F1'] = (thresh_grid[idx_f1], P_arr[idx_f1], R_arr[idx_f1], F_arr[idx_f1])

# Recall >= 0.90 with max precision
mask_rec  = R_arr >= 0.90
if mask_rec.any():
    idx_rec = np.where(mask_rec)[0][np.argmax(P_arr[mask_rec])]
    goals['Recall >= 0.90'] = (
        thresh_grid[idx_rec], P_arr[idx_rec], R_arr[idx_rec], F_arr[idx_rec])

# Precision >= 0.90 with max recall
mask_prec = P_arr >= 0.90
if mask_prec.any():
    idx_prec = np.where(mask_prec)[0][np.argmax(R_arr[mask_prec])]
    goals['Precision >= 0.90'] = (
        thresh_grid[idx_prec], P_arr[idx_prec], R_arr[idx_prec], F_arr[idx_prec])

# Balanced (P closest to R)
idx_bal   = np.argmin(np.abs(P_arr - R_arr))
goals['Balanced (P=R)'] = (
    thresh_grid[idx_bal], P_arr[idx_bal], R_arr[idx_bal], F_arr[idx_bal])

plt.figure(figsize=(12, 6))
plt.plot(thresh_grid, P_arr, color=COLORS['primary'],  linewidth=2, label='Precision')
plt.plot(thresh_grid, R_arr, color=COLORS['danger'],   linewidth=2, label='Recall')
plt.plot(thresh_grid, F_arr, color=COLORS['success'],  linewidth=2, label='F1')

marker_colors = [COLORS['success'], COLORS['danger'],
                 COLORS['primary'], COLORS['tertiary']]
for (goal_name, result), mc in zip(goals.items(), marker_colors):
    if result:
        t, p, r, f = result
        plt.axvline(t, color=mc, linestyle=':', linewidth=1.5,
                    label=f'{goal_name} @ t={t:.2f}')

plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold Tuning for Different Business Goals')
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print(f'{"Goal":<25} {"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 68)
for goal_name, result in goals.items():
    if result:
        t, p, r, f = result
        print(f'{goal_name:<25} {t:>10.3f} {p:>10.4f} {r:>10.4f} {f:>10.4f}')

## 7. Multiclass Precision-Recall

In [ ]:
np.random.seed(42)
X_mc, y_mc = make_classification(
    n_samples=600, n_features=12, n_informative=7,
    n_redundant=3, n_classes=4, n_clusters_per_class=1,
    random_state=42
)
X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc
)
sc_mc = StandardScaler()
X_mc_tr_sc = sc_mc.fit_transform(X_mc_tr)
X_mc_te_sc = sc_mc.transform(X_mc_te)

mc_model = LogisticRegression(max_iter=500, random_state=42)
mc_model.fit(X_mc_tr_sc, y_mc_tr)
y_mc_pred = mc_model.predict(X_mc_te_sc)

class_names = ['Class A','Class B','Class C','Class D']

# Per-class metrics
p_per = precision_score(y_mc_te, y_mc_pred, average=None)
r_per = recall_score(   y_mc_te, y_mc_pred, average=None)
f_per = f1_score(       y_mc_te, y_mc_pred, average=None)

# Averaged metrics
averaging_types = ['macro','micro','weighted']
avg_results = {}
for avg in averaging_types:
    avg_results[avg] = {
        'Precision': precision_score(y_mc_te, y_mc_pred, average=avg),
        'Recall':    recall_score(   y_mc_te, y_mc_pred, average=avg),
        'F1':        f1_score(       y_mc_te, y_mc_pred, average=avg),
    }

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per-class bar chart
x_pos = np.arange(len(class_names))
width = 0.25
axes[0].bar(x_pos - width, p_per, width, label='Precision',
            color=COLORS['primary'],   edgecolor='white')
axes[0].bar(x_pos,          r_per, width, label='Recall',
            color=COLORS['danger'],    edgecolor='white')
axes[0].bar(x_pos + width,  f_per, width, label='F1',
            color=COLORS['success'],   edgecolor='white')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(class_names)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Score')
axes[0].set_title('Per-Class Precision / Recall / F1')
axes[0].legend()
for i, (p,r,f) in enumerate(zip(p_per, r_per, f_per)):
    axes[0].text(i-width, p+0.02, f'{p:.2f}', ha='center', fontsize=7)
    axes[0].text(i,       r+0.02, f'{r:.2f}', ha='center', fontsize=7)
    axes[0].text(i+width, f+0.02, f'{f:.2f}', ha='center', fontsize=7)

# Averaging comparison
avg_df = pd.DataFrame(avg_results).T
avg_df.plot(kind='bar', ax=axes[1],
            color=[COLORS['primary'], COLORS['danger'], COLORS['success']],
            edgecolor='white', rot=0)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score')
axes[1].set_title('Macro vs Micro vs Weighted Averaging')
axes[1].legend()

plt.suptitle('Multiclass Precision-Recall-F1 (4 classes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Averaging types:')
print('  macro    : simple average across classes (ignores class imbalance)')
print('  micro    : aggregate TP/FP/FN globally before computing')
print('  weighted : weighted by class support (sample count)')
print()
print(classification_report(y_mc_te, y_mc_pred, target_names=class_names))

## 8. Imbalanced Classes — Why F1 Matters

In [ ]:
np.random.seed(42)
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, weights=[0.95, 0.05], random_state=42
)
X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)
sc_imb = StandardScaler()
X_imb_tr_sc = sc_imb.fit_transform(X_imb_tr)
X_imb_te_sc = sc_imb.transform(X_imb_te)

print(f'Class balance: {np.bincount(y_imb)}  (0=Majority, 1=Minority 5%)')

y_all_neg  = np.zeros(len(y_imb_te), dtype=int)

lr_no_weight = LogisticRegression(max_iter=500, random_state=42)
lr_no_weight.fit(X_imb_tr_sc, y_imb_tr)

lr_balanced = LogisticRegression(max_iter=500, class_weight='balanced',
                                  random_state=42)
lr_balanced.fit(X_imb_tr_sc, y_imb_tr)

scenarios = [
    ('All-Negative (baseline)', y_all_neg),
    ('LR no class_weight',      lr_no_weight.predict(X_imb_te_sc)),
    ('LR balanced weight',      lr_balanced.predict(X_imb_te_sc)),
]

rows_imb = []
for name, y_pred_imb in scenarios:
    rows_imb.append({
        'Model': name,
        'Accuracy':  round(np.mean(y_pred_imb == y_imb_te), 4),
        'Precision': round(precision_score(y_imb_te, y_pred_imb, zero_division=0), 4),
        'Recall':    round(recall_score(   y_imb_te, y_pred_imb, zero_division=0), 4),
        'F1':        round(f1_score(       y_imb_te, y_pred_imb, zero_division=0), 4),
    })

df_imb = pd.DataFrame(rows_imb)
print('\nMetric comparison on 95/5 imbalanced dataset:')
print(df_imb.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred_imb) in zip(axes, scenarios):
    cm_imb = confusion_matrix(y_imb_te, y_pred_imb)
    ConfusionMatrixDisplay(cm_imb,
                            display_labels=['Majority','Minority']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    p = precision_score(y_imb_te, y_pred_imb, zero_division=0)
    r = recall_score(   y_imb_te, y_pred_imb, zero_division=0)
    f = f1_score(       y_imb_te, y_pred_imb, zero_division=0)
    ax.set_title(f'{name}\nP={p:.3f} R={r:.3f} F1={f:.3f}', fontsize=9)

plt.suptitle('Imbalanced Classes — F1 Reveals What Accuracy Hides',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('All-Negative: 95% accuracy but F1=0 -> completely useless for minority class!')
print('class_weight="balanced" -> much better Recall and F1.')

## 9. Real-World Offline-Safe Dataset — Fraud Detection

In [ ]:
np.random.seed(42)
X_fraud, y_fraud = make_classification(
    n_samples=1000, n_features=12, n_informative=7,
    n_redundant=3, weights=[0.85, 0.15],
    random_state=42
)
feat_names_fraud = ['TransAmount','MerchantCat','TimeOfDay','CardAge',
                    'NumTransDay','AvgSpend','DeviceType','GeoDistance',
                    'PrevFraud','CVVMatch','BillingMatch','VelocityScore']

print(f'Dataset shape  : {X_fraud.shape}')
print(f'Class balance  : {np.bincount(y_fraud)}  (0=Legit, 1=Fraud)')

X_fr_tr, X_fr_te, y_fr_tr, y_fr_te = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)
sc_fr = StandardScaler()
X_fr_tr_sc = sc_fr.fit_transform(X_fr_tr)
X_fr_te_sc = sc_fr.transform(X_fr_te)

fraud_models = {
    'Logistic Reg (bal)': LogisticRegression(max_iter=500, class_weight='balanced',
                                              random_state=42),
    'Random Forest':      RandomForestClassifier(100, class_weight='balanced',
                                                 random_state=42),
    'Gradient Boosting':  GradientBoostingClassifier(100, learning_rate=0.1,
                                                      random_state=42),
}

rows_fraud = []
for name, clf in fraud_models.items():
    clf.fit(X_fr_tr_sc, y_fr_tr)
    yp   = clf.predict(X_fr_te_sc)
    prob = clf.predict_proba(X_fr_te_sc)[:, 1]
    rows_fraud.append({
        'Model':     name,
        'Precision': round(precision_score(y_fr_te, yp), 4),
        'Recall':    round(recall_score(   y_fr_te, yp), 4),
        'F1':        round(f1_score(       y_fr_te, yp), 4),
        'F2':        round(fbeta_score(    y_fr_te, yp, beta=2), 4),
        'AP':        round(average_precision_score(y_fr_te, prob), 4),
        'AUC':       round(roc_auc_score(  y_fr_te, prob), 4),
    })

df_fraud = pd.DataFrame(rows_fraud)
print('\nFraud Detection Model Comparison:')
print(df_fraud.to_string(index=False))

# Metric bar comparison
metric_cols = ['Precision','Recall','F1','F2','AP','AUC']
x_pos = np.arange(len(metric_cols))
width = 0.25

plt.figure(figsize=(13, 6))
for i, (_, row) in enumerate(df_fraud.iterrows()):
    plt.bar(x_pos + i*width,
            [row[m] for m in metric_cols],
            width, label=row['Model'],
            color=MODEL_COLORS[i], edgecolor='white', alpha=0.85)
plt.xticks(x_pos + width, metric_cols, fontsize=10)
plt.ylabel('Score')
plt.ylim(0, 1.15)
plt.title('Fraud Detection — All Metrics Compared')
plt.legend()
plt.tight_layout()
plt.show()

print()
print('F2 score used here because missing fraud (FN) is worse than false alarms (FP).')

## 10. Metric Selection Guide

In [ ]:
print('=' * 66)
print('         PRECISION / RECALL / F1 - SELECTION GUIDE')
print('=' * 66)

guide = [
    ('Max Precision',     'Spam filter, drug approval — FP is costly'),
    ('Max Recall',        'Cancer screening, fraud — FN is costly'),
    ('F1 (beta=1)',       'General; equal weight to P and R'),
    ('F2 (beta=2)',       'Recall 2x more important (medical, safety)'),
    ('F0.5 (beta=0.5)',   'Precision 2x more important (legal, finance)'),
    ('Avg Precision (AP)','Imbalanced data, ranking quality of scores'),
    ('Macro F1',          'Multiclass, all classes equally important'),
    ('Weighted F1',       'Multiclass, weight by class frequency'),
    ('Micro F1',          'Multiclass, aggregate globally'),
]
print(f'  {"Use Case":<28} {"Recommended Metric"}')
print('  ' + '-'*58)
for use_case, metric in guide:
    print(f'  {use_case:<28} {metric}')

print()
print('Threshold tuning targets:')
targets = [
    ('Maximize F1',        'argmax(F1) across thresholds'),
    ('Recall >= 0.95',     'Find highest-P threshold with R>=0.95'),
    ('Precision >= 0.90',  'Find highest-R threshold with P>=0.90'),
    ('Equal P and R',      'Find where Precision curve crosses Recall'),
]
for goal, method in targets:
    print(f'  Goal: {goal:<28} -> {method}')

print('=' * 66)

## 11. Final Summary & Key Takeaways

In [ ]:
print('=' * 66)
print('   PRECISION, RECALL & F1 - KEY TAKEAWAYS')
print('=' * 66)
takeaways = [
    ('Precision',      'TP/(TP+FP) — quality of positive predictions'),
    ('Recall',         'TP/(TP+FN) — coverage of actual positives'),
    ('Tradeoff',       'Increasing one usually decreases the other'),
    ('F1',             'Harmonic mean; penalizes extreme imbalance between P and R'),
    ('F-beta',         'beta>1 favors Recall; beta<1 favors Precision'),
    ('PR Curve',       'Plots P vs R at every threshold; area = AP'),
    ('AP',             'Average Precision; use for imbalanced classes'),
    ('Iso-F1 lines',   'Diagonal curves on PR plot where F1 is constant'),
    ('Imbalanced',     'F1 exposes what accuracy hides on skewed classes'),
    ('class_weight',   'balanced weight gives minority class fair treatment'),
    ('Threshold',      'Tune threshold for your specific P/R goal'),
    ('Multiclass',     'Use macro/micro/weighted depending on class balance'),
    ('PR vs ROC',      'PR curve is more informative when positive class is rare'),
]
for topic, detail in takeaways:
    print(f'  OK  {topic:<18}  ->  {detail}')
print('=' * 66)

---
## Practice Exercises

1. **Custom cost function** — assign cost 10 to FN and 1 to FP, find the threshold minimizing total cost.
2. **SMOTE** — apply oversampling to the fraud dataset and compare F1 before and after.
3. **F-beta sweep** — vary beta from 0.1 to 5 and plot the optimal threshold for each beta.
4. **PR curve for all models** — plot PR curves for all fraud models on the same axes.
5. **Cross-validated F1** — use `cross_val_score(scoring='f1')` and compare model stability.
6. **Implement F1 from scratch** using only NumPy and verify it matches sklearn.

---
*Notebook created for the ML Repository — Precision, Recall & F1 module.*